## google madlad

In [1]:
import os
import glob
import re
import random
import gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import nltk
from tqdm import tqdm

In [ ]:


def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

nltk.download('wordnet', quiet=True)

data_dir = "Dataset_WMT/"
model_name = "google/madlad400-10b-mt"
device = "cuda" if torch.cuda.is_available() else "cpu"
output_path = "madlad400_10b_mt_evaluation_results_I.csv"

BATCH_SIZE = 64  

LANG_MAP = {
    "Assamese": "as",
    "Khasi": "kha",
    "Manipuri": " mni",       
    "Meitei-Mayek": "mni",   
    "Mizo": "lus",
    "Nyishi": "brx",  
    "Bodo": "brx",      
    "Kokborok": "brx",        
    "Karbi": "lus",           
    "Nagamese": "as",       
    "Tagin": "brx"  
 }

print("Loading evaluation metrics...")
bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

print(f"Loading model {model_name} on {device} in BF16...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    use_safetensors=True, 
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(device)

if hasattr(torch, "compile") and device == "cuda":
    print("Compiling model for faster inference execution...")
    model = torch.compile(model)

def determine_lang_info(filename):
    for lang, lang_code in LANG_MAP.items():
        if lang.lower() in filename.lower():
            return lang, lang_code
    return None, None

def detect_columns(df, filename):
    cols = [str(c).strip() for c in df.columns]    
    is_numeric_header = all(c.isdigit() for c in cols)
    
    if is_numeric_header:
        print(f" No named headers detected (numeric headers found). Using filename heuristics...")
        filename_lower = filename.lower()
        lang_name, _ = determine_lang_info(filename)
        
        if lang_name and filename_lower.find('eng') < filename_lower.find(lang_name.lower()):
            en_col, src_col = df.columns[0], df.columns[1]
        else:
            src_col, en_col = df.columns[0], df.columns[1]
            
        return src_col, en_col

    df.columns = cols  
    if len(cols) < 2:
        raise ValueError("File does not contain at least 2 columns.")
        
    en_indicators = ['english', 'en', 'eng', 'English']
    en_col = None
    for col in cols:
        if col.lower() in en_indicators:
            en_col = col
            break
    if not en_col:
        for col in cols:
            if 'eng' in col.lower() or col.lower() == 'en':
                en_col = col
                break
                
    if not en_col:
        print(" Unclear column names. Falling back to character distribution analysis...")
        ascii_scores = {}
        for col in cols:
            sample = df[col].dropna().head(100).astype(str)
            if sample.empty:
                ascii_scores[col] = 0
                continue
            total_chars = sum(len(s) for s in sample)
            ascii_chars = sum(len(re.findall(r'[A-Za-z0-9\s.,!?]', s)) for s in sample)
            ascii_scores[col] = ascii_chars / total_chars if total_chars > 0 else 0
        en_col = max(ascii_scores, key=ascii_scores.get)
        
    other_cols = [c for c in cols if c != en_col]
    src_col = other_cols[0] 
    return src_col, en_col

def translate_batch_generator(texts, tgt_code="en", batch_size=32):
    prefix = f"<2{tgt_code}> "
    for i in range(0, len(texts), batch_size):
        batch = [prefix + str(text) for text in texts[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.inference_mode():
            generated_tokens = model.generate(**inputs, max_length=256, num_beams=1, do_sample=False, early_stopping=True)
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        yield decoded
        
def save_incremental_results(results_dict, out_path):
    if not results_dict:
        return
    res_df = pd.DataFrame.from_dict(results_dict, orient='index')
    if len(res_df) > 1:
        calc_df = res_df.drop('OVERALL (Macro Avg)', errors='ignore')
        overall_avg = calc_df.mean(numeric_only=True).round(2)
        res_df.loc['OVERALL (Macro Avg)'] = overall_avg
        
    res_df.to_csv(out_path)
    print(f"\n--- Live metrics written back to: {out_path} ---")

# --- Process Datasets ---
all_files = glob.glob(os.path.join(data_dir, "*.csv")) + glob.glob(os.path.join(data_dir, "*.xlsx"))
language_results = {}

print(f"Found {len(all_files)} files to process.")

for file_path in all_files:
    filename = os.path.basename(file_path)
    lang_name, src_code = determine_lang_info(filename)
    
    if not lang_name:
        print(f"Skipping {filename}: Could not determine target language grouping.")
        continue
    if not src_code:
        print(f"Skipping {filename}: Language '{lang_name}' is not supported by MADLAD-400.")
        continue   
    print(f"\nProcessing {lang_name} ({filename})...")   
    try:
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = pd.read_excel(file_path)     
        src_col, en_col = detect_columns(df, filename)
        print(f"-> Mapped input columns: [Source: '{src_col}'] to [English Target: '{en_col}']")    
        df = df[[src_col, en_col]].dropna().astype(str)
        src_sentences = df[src_col].tolist()
        ref_sentences = df[en_col].tolist()
        if not src_sentences:
            print(f" Empty dataset for {filename} after scrubbing null elements.")
            continue 
        print(f"Translating {len(src_sentences)} sentences using optimized batching...")
        hyp_sentences = []
        
        for batch_hyp in tqdm(translate_batch_generator(src_sentences, tgt_code="en", batch_size=BATCH_SIZE), 
                              total=(len(src_sentences) + (BATCH_SIZE - 1)) // BATCH_SIZE, desc="Batches"):
            hyp_sentences.extend(batch_hyp)
                
        # Save predictions
        prediction_df = df.copy()
        prediction_df["Prediction"] = hyp_sentences
        
        prediction_output_path = f"{os.path.splitext(filename)[0]}_predictions.csv"
        prediction_df.to_csv(prediction_output_path, index=False)
        print(f"Predictions saved to: {prediction_output_path}")  
        print("Computing metrics safely...")
        bleu_ref = [[ref] for ref in ref_sentences]
        
        scores = {"BLEU": None, "chrF": None, "ROUGE-1": None, "ROUGE-L": None, "METEOR": None, "BERTScore-F1": None}
        
        try:
            bleu = bleu_metric.compute(predictions=hyp_sentences, references=bleu_ref)
            scores["BLEU"] = round(bleu['bleu'] * 100, 2)
        except Exception as me:
            print(f"  [Error] Failed calculating BLEU: {me}")
            
        try:
            chrf = chrf_metric.compute(predictions=hyp_sentences, references=bleu_ref)
            scores["chrF"] = round(chrf['score'], 2)
        except Exception as me:
            print(f"  [Error] Failed calculating chrF: {me}")
            
        try:
            rouge = rouge_metric.compute(predictions=hyp_sentences, references=ref_sentences)
            scores["ROUGE-1"] = round(rouge['rouge1'] * 100, 2)
            scores["ROUGE-L"] = round(rouge['rougeL'] * 100, 2)
        except Exception as me:
            print(f"  [Error] Failed calculating ROUGE: {me}")
            
        try:
            meteor = meteor_metric.compute(predictions=hyp_sentences, references=ref_sentences)
            scores["METEOR"] = round(meteor['meteor'] * 100, 2)
        except Exception as me:
            print(f"  [Error] Failed calculating METEOR: {me}")
            
        try:
            # Accelerated BERTScore by introducing batching and GPU pinning directly into target evaluation parameters
            bertscore = bertscore_metric.compute(
                predictions=hyp_sentences, 
                references=ref_sentences, 
                lang="en", 
                batch_size=BATCH_SIZE, 
                device=device
            )
            scores["BERTScore-F1"] = round(sum(bertscore['f1']) / len(bertscore['f1']) * 100, 2)
        except Exception as me:
            print(f"  [Error] Failed calculating BERTScore: {me}")
            
        language_results[lang_name] = scores
        save_incremental_results(language_results, output_path)
        
    except Exception as e:
        print(f" Critical error occurred while processing {lang_name} ({filename}): {e}")
    finally:
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
        print("Moving to next language file...")

if language_results:
    final_df = pd.DataFrame.from_dict(language_results, orient='index')
    calc_df = final_df.drop('OVERALL (Macro Avg)', errors='ignore')
    overall_avg = calc_df.mean(numeric_only=True).round(2)
    final_df.loc['OVERALL (Macro Avg)'] = overall_avg  
    print("\n" + "="*60)
    print(" FINAL EVALUATION SUMMARY ")
    print("="*60)
    print(final_df.to_markdown())
else:
    print("No languages were successfully processed.")

/home/hbbg/environments/wmt/lib/python3.10/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Loading evaluation metrics...


[nltk_data] Downloading package wordnet to /home/hbbg/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/hbbg/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/hbbg/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading model google/madlad400-10b-mt on cpu in BF16...


tokenizer_config.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

...

model-00009-of-00009.safetensors:   0%|          | 0.00/3.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Found 11 files to process.

Processing Assamese (English-Assamese Training Data 2026.csv)...
-> Mapped input columns: [Source: 'as'] to [English Target: 'en']
Translating 54000 sentences using optimized batching...


Batches:   0%|                                                                                                                                                                     | 0/844 [00:00<?, ?it/s]/home/hbbg/environments/wmt/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:563: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
Batches:   0%|▏                                                                                                                                                        | 1/844 [01:40<23:34:34, 100.68s/it]

## Tranliterated text to English Translation google/madlad

In [ ]:
# import os
# import glob
# import re
# import random
# import gc
# import numpy as np
# import pandas as pd
# import torch
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# import evaluate
# import nltk
# from tqdm import tqdm

# from ai4bharat.transliteration import XlitEngine


# def set_seed(seed=42):
#     random.seed(seed)
#     os.environ['PYTHONHASHSEED'] = str(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed) 
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(42)

# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True

# nltk.download('wordnet', quiet=True)

# data_dir = "native"
# model_name = "google/madlad400-3b-mt"
# device = "cuda" if torch.cuda.is_available() else "cpu"
# output_path = "madlad400_3b_mt_evaluation_results_native_english_transliteration.csv"

# BATCH_SIZE = 32  


# LANG_MAP = {
#     "Assamese": {"madlad": "as", "xlit": "as"},
#     "Khasi": {"madlad": "kha", "xlit": None},
#     "Manipuri": {"madlad": "mni", "xlit": "mni"},       
#     "Meitei-Mayek": {"madlad": "mni", "xlit": "mni"}
#     "Mizo": {"madlad": "lus", "xlit": None},
#     "Nyishi": {"madlad": "brx", "xlit": None},  
#     "Bodo": {"madlad": "brx", "xlit": "brx"},      
#     "Kokborok": {"madlad": "brx", "xlit": None},        
#     "Karbi": {"madlad": "lus", "xlit": None},            
#     "Nagamese": {"madlad": "as", "xlit": None},        
#     "Tagin": {"madlad": "brx", "xlit": None}  
# }

# print("Loading evaluation metrics...")
# bleu_metric = evaluate.load("bleu")
# chrf_metric = evaluate.load("chrf")
# rouge_metric = evaluate.load("rouge")
# meteor_metric = evaluate.load("meteor")
# bertscore_metric = evaluate.load("bertscore")

# print(f"Loading model {model_name} on {device} in BF16...")
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model = AutoModelForSeq2SeqLM.from_pretrained(
#     model_name, 
#     use_safetensors=True, 
#     torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
#     low_cpu_mem_usage=True
# ).to(device)

# if hasattr(torch, "compile") and device == "cuda":
#     print("Compiling model for faster inference execution...")
#     model = torch.compile(model)

# def determine_lang_info(filename):
#     for lang, config in LANG_MAP.items():
#         if lang.lower() in filename.lower():
#             return lang, config
#     return None, None

# def detect_columns(df, filename):
#     cols = [str(c).strip() for c in df.columns]    
#     is_numeric_header = all(c.isdigit() for c in cols)
    
#     if is_numeric_header:
#         print(f" No named headers detected (numeric headers found). Using filename heuristics...")
#         filename_lower = filename.lower()
#         lang_name, _ = determine_lang_info(filename)
        
#         if lang_name and filename_lower.find('eng') < filename_lower.find(lang_name.lower()):
#             en_col, src_col = df.columns[0], df.columns[1]
#         else:
#             src_col, en_col = df.columns[0], df.columns[1]
            
#         return src_col, en_col

#     df.columns = cols  
#     if len(cols) < 2:
#         raise ValueError("File does not contain at least 2 columns.")
        
#     en_indicators = ['english', 'en', 'eng', 'English']
#     en_col = None
#     for col in cols:
#         if col.lower() in en_indicators:
#             en_col = col
#             break
#     if not en_col:
#         for col in cols:
#             if 'eng' in col.lower() or col.lower() == 'en':
#                 en_col = col
#                 break
                
#     if not en_col:
#         print(" Unclear column names. Falling back to character distribution analysis...")
#         ascii_scores = {}
#         for col in cols:
#             sample = df[col].dropna().head(100).astype(str)
#             if sample.empty:
#                 ascii_scores[col] = 0
#                 continue
#             total_chars = sum(len(s) for s in sample)
#             ascii_chars = sum(len(re.findall(r'[A-Za-z0-9\s.,!?]', s)) for s in sample)
#             ascii_scores[col] = ascii_chars / total_chars if total_chars > 0 else 0
#         en_col = max(ascii_scores, key=ascii_scores.get)
        
#     other_cols = [c for c in cols if c != en_col]
#     src_col = other_cols[0] 
#     return src_col, en_col

# def run_transliteration(texts, xlit_code, batch_size=32):
#     """Transliterates Native Indic scripts into Latin/Roman script via AI4Bharat using batch sizing."""
#     if not xlit_code:
#         return texts
    
#     print(f"-> Transliterating native text to Latin script via AI4Bharat using code: '{xlit_code}'")
#     try:
#         import argparse
#         import inspect
#         if hasattr(torch, "serialization") and hasattr(torch.serialization, "add_safe_globals"):
#             torch.serialization.add_safe_globals([argparse.Namespace])
            
#         e = XlitEngine(beam_width=4, rescore=False, src_script_type="indic")
#         transliterated_texts = []
        
#         # Resolve the actual method used by the active backend wrapper class
#         translit_func = getattr(e, "translit_sentence", None) or getattr(e, "transliterate_sentence", None)
#         if translit_func is None:
#             raise AttributeError("Could not locate a valid transliteration method on XlitEngine.")
            
#         # Inspect parameter layout requirements
#         sig = inspect.signature(translit_func)
#         params = sig.parameters
        
#         # Break dataset into chunks matching batch size allocations
#         for i in tqdm(range(0, len(texts), batch_size), desc="Transliterating Blocks"):
#             batch = texts[i:i + batch_size]
            
#             if 'lang_code' in params:
#                 out_batch = [translit_func(text, lang_code=xlit_code) for text in batch]
#             elif 'lang' in params:
#                 out_batch = [translit_func(text, lang=xlit_code) for text in batch]
#             else:
#                 out_batch = [translit_func(text) for text in batch]
                
#             transliterated_texts.extend(out_batch)
            
#         return transliterated_texts
        
#     except Exception as ex:
#         print(f" [Error] Transliteration engine failed: {ex}. Using original text.")
#         return texts
        
# def translate_batch_generator(texts, tgt_code="en", batch_size=32):
#     prefix = f"<2{tgt_code}> "
#     for i in range(0, len(texts), batch_size):
#         batch = [prefix + str(text) for text in texts[i:i+batch_size]]
#         inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
#         with torch.inference_mode():
#             generated_tokens = model.generate(**inputs, max_length=256, num_beams=1, do_sample=False, early_stopping=True)
#         decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
#         yield decoded
        
# def save_incremental_results(results_dict, out_path):
#     if not results_dict:
#         return
#     res_df = pd.DataFrame.from_dict(results_dict, orient='index')
#     if len(res_df) > 1:
#         calc_df = res_df.drop('OVERALL (Macro Avg)', errors='ignore')
#         overall_avg = calc_df.mean(numeric_only=True).round(2)
#         res_df.loc['OVERALL (Macro Avg)'] = overall_avg
        
#     res_df.to_csv(out_path)
#     print(f"\n--- Live metrics written back to: {out_path} ---")

# all_files = glob.glob(os.path.join(data_dir, "*.csv")) + glob.glob(os.path.join(data_dir, "*.xlsx"))
# language_results = {}

# print(f"Found {len(all_files)} files to process.")

# for file_path in all_files:
#     filename = os.path.basename(file_path)
#     lang_name, config = determine_lang_info(filename)
    
#     if not lang_name:
#         print(f"Skipping {filename}: Could not determine target language grouping.")
#         continue
        
#     src_code = config["madlad"]
#     xlit_code = config["xlit"]
    
#     if not src_code:
#         print(f"Skipping {filename}: Language '{lang_name}' is not supported by MADLAD-400.")
#         continue   
#     print(f"\nProcessing {lang_name} ({filename})...")   
#     try:
#         if file_path.endswith('.csv'):
#             df = pd.read_csv(file_path)
#         else:
#             df = pd.read_excel(file_path)     
#         src_col, en_col = detect_columns(df, filename)
#         print(f"-> Mapped input columns: [Source: '{src_col}'] to [English Target: '{en_col}']")    
#         df = df[[src_col, en_col]].dropna().astype(str)
        
#         raw_src_sentences = df[src_col].tolist()
#         ref_sentences = df[en_col].tolist()
#         if not raw_src_sentences:
#             print(f" Empty dataset for {filename} after scrubbing null elements.")
#             continue 
            
#         src_sentences = run_transliteration(raw_src_sentences, xlit_code, batch_size=BATCH_SIZE)
        
#         print(f"Translating {len(src_sentences)} sentences using optimized batching...")
#         hyp_sentences = []
        
#         for batch_hyp in tqdm(translate_batch_generator(src_sentences, tgt_code="en", batch_size=BATCH_SIZE), 
#                               total=(len(src_sentences) + (BATCH_SIZE - 1)) // BATCH_SIZE, desc="Batches"):
#             hyp_sentences.extend(batch_hyp)
                
#         prediction_df = pd.DataFrame({
#             'Source_Language_Original': raw_src_sentences,
#             'Source_Language_Transliterated': src_sentences,
#             'Ground_Truth_English': ref_sentences,
#             'Prediction': hyp_sentences
#         })
        
#         prediction_output_path = f"{os.path.splitext(filename)[0]}_predictions.csv"
#         prediction_df.to_csv(prediction_output_path, index=False)
#         print(f"Predictions saved to: {prediction_output_path}")  
#         print("Computing metrics safely...")
#         bleu_ref = [[ref] for ref in ref_sentences]
        
#         scores = {"BLEU": None, "chrF": None, "ROUGE-1": None, "ROUGE-L": None, "METEOR": None, "BERTScore-F1": None}
        
#         try:
#             bleu = bleu_metric.compute(predictions=hyp_sentences, references=bleu_ref)
#             scores["BLEU"] = round(bleu['bleu'] * 100, 2)
#         except Exception as me:
#             print(f"  [Error] Failed calculating BLEU: {me}")
            
#         try:
#             chrf = chrf_metric.compute(predictions=hyp_sentences, references=bleu_ref)
#             scores["chrF"] = round(chrf['score'], 2)
#         except Exception as me:
#             print(f"  [Error] Failed calculating chrF: {me}")
            
#         try:
#             rouge = rouge_metric.compute(predictions=hyp_sentences, references=ref_sentences)
#             scores["ROUGE-1"] = round(rouge['rouge1'] * 100, 2)
#             scores["ROUGE-L"] = round(rouge['rougeL'] * 100, 2)
#         except Exception as me:
#             print(f"  [Error] Failed calculating ROUGE: {me}")
            
#         try:
#             meteor = meteor_metric.compute(predictions=hyp_sentences, references=ref_sentences)
#             scores["METEOR"] = round(meteor['meteor'] * 100, 2)
#         except Exception as me:
#             print(f"  [Error] Failed calculating METEOR: {me}")
            
#         try:
#             bertscore = bertscore_metric.compute(
#                 predictions=hyp_sentences, 
#                 references=ref_sentences, 
#                 lang="en", 
#                 batch_size=BATCH_SIZE, 
#                 device=device
#             )
#             scores["BERTScore-F1"] = round(sum(bertscore['f1']) / len(bertscore['f1']) * 100, 2)
#         except Exception as me:
#             print(f"  [Error] Failed calculating BERTScore: {me}")
            
#         language_results[lang_name] = scores
#         save_incremental_results(language_results, output_path)
        
#     except Exception as e:
#         print(f" Critical error occurred while processing {lang_name} ({filename}): {e}")
#     finally:
#         gc.collect()
#         if device == "cuda":
#             torch.cuda.empty_cache()
#         print("Moving to next language file...")

# # --- Final Output Block ---
# if language_results:
#     final_df = pd.DataFrame.from_dict(language_results, orient='index')
#     calc_df = final_df.drop('OVERALL (Macro Avg)', errors='ignore')
#     overall_avg = calc_df.mean(numeric_only=True).round(2)
#     final_df.loc['OVERALL (Macro Avg)'] = overall_avg  
#     print("\n" + "="*60)
#     print(" FINAL EVALUATION SUMMARY ")
#     print("="*60)
#     print(final_df.to_markdown())
# else:
#     print("No languages were successfully processed.")

## back transliteration

In [ ]:
# import os
# import glob
# import re
# import random
# import gc
# import numpy as np
# import pandas as pd
# import torch
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# import evaluate
# import nltk
# from tqdm import tqdm
# from concurrent.futures import ProcessPoolExecutor

# from ai4bharat.transliteration import XlitEngine

# def set_seed(seed=42):
#     random.seed(seed)
#     os.environ['PYTHONHASHSEED'] = str(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed) 
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(42)

# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True

# nltk.download('wordnet', quiet=True)

# data_dir = "backtransliteration"
# model_name = "google/madlad400-3b-mt"
# device = "cuda" if torch.cuda.is_available() else "cpu"
# output_path = "madlad400_3b_mt_evaluation_results_kokborok_bodo_transliteration.csv"

# BATCH_SIZE = 32  

# # Updated Mapping specifically handling Romanized Nagamese mapped to Assamese target script
# # LANG_MAP = {
# #     "Nagamese": {"madlad": "as", "xlit": "as"},  # Redirected to use Assamese engine configurations
# # }

# LANG_MAP = {
#     "Kokborok": {"madlad": "brx", "xlit": "brx"},  # Redirected to use Bode engine configurations
# }

# print("Loading evaluation metrics...")
# bleu_metric = evaluate.load("bleu")
# chrf_metric = evaluate.load("chrf")
# rouge_metric = evaluate.load("rouge")
# meteor_metric = evaluate.load("meteor")
# bertscore_metric = evaluate.load("bertscore")

# print(f"Loading model {model_name} on {device} in BF16...")
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model = AutoModelForSeq2SeqLM.from_pretrained(
#     model_name, 
#     use_safetensors=True, 
#     torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
#     low_cpu_mem_usage=True
# ).to(device)

# if hasattr(torch, "compile") and device == "cuda":
#     print("Compiling model for faster inference execution...")
#     model = torch.compile(model)

# def determine_lang_info(filename):
#     for lang, config in LANG_MAP.items():
#         if lang.lower() in filename.lower():
#             return lang, config
#     return None, None

# def detect_columns(df, filename):
#     cols = [str(c).strip() for c in df.columns]    
#     if len(cols) < 2:
#         raise ValueError("File does not contain at least 2 columns.")
    
#     df.columns = cols
#     en_indicators = ['english', 'en', 'eng', 'English']
#     en_col = next((col for col in cols if col.lower() in en_indicators), None)
    
#     if not en_col:
#         en_col = cols[1] 
#     src_col = [c for c in cols if c != en_col][0]
#     return src_col, en_col

# def process_single_roman_to_native(args):
#     text, xlit_code = args
#     try:
#         import argparse
#         import torch
#         if hasattr(torch, "serialization") and hasattr(torch.serialization, "add_safe_globals"):
#             torch.serialization.add_safe_globals([argparse.Namespace])
            
#         from ai4bharat.transliteration import XlitEngine
#         e = XlitEngine(xlit_code, beam_width=4, rescore=True, src_script_type="roman")
#         out = e.translit_sentence(str(text))
#         if isinstance(out, dict) and xlit_code in out:
#             return out[xlit_code]
#         return str(text)
#     except:
#         return str(text)

# def run_transliteration(texts, xlit_code):
#     if not xlit_code:
#         return texts
#     print(f"-> Converting Romanized Nagamese to Assamese script safely...")
    
#     try:
#         import argparse
#         import torch
#         if hasattr(torch, "serialization") and hasattr(torch.serialization, "add_safe_globals"):
#             torch.serialization.add_safe_globals([argparse.Namespace])
#         from ai4bharat.transliteration import XlitEngine
#         e = XlitEngine(xlit_code, beam_width=4, rescore=True, src_script_type="roman")
#         transliterated_texts = []
#         for text in tqdm(texts, desc="Roman -> Bodo Script"):
#             try:
#                 out = e.translit_sentence(str(text))
#                 if isinstance(out, dict) and xlit_code in out:
#                     transliterated_texts.append(out[xlit_code])
#                 else:
#                     transliterated_texts.append(str(text))
#             except Exception:
#                 transliterated_texts.append(str(text))     
#         return transliterated_texts
#     except Exception as ex:
#         print(f" [Error] Transliteration engine failed to start: {ex}. Using original text.")
#         return texts  
        
# def translate_batch_generator(texts, tgt_code="en", batch_size=32):
#     prefix = f"<2{tgt_code}> "
#     for i in range(0, len(texts), batch_size):
#         batch = [prefix + str(text) for text in texts[i:i+batch_size]]
#         inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
#         with torch.inference_mode():
#             generated_tokens = model.generate(**inputs, max_length=256, num_beams=1, do_sample=False, early_stopping=True)
#         decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
#         yield decoded
        
# def save_incremental_results(results_dict, out_path):
#     if not results_dict:
#         return
#     res_df = pd.DataFrame.from_dict(results_dict, orient='index')
#     res_df.to_csv(out_path)
#     print(f"\n--- Live metrics written to: {out_path} ---")


# all_files = glob.glob(os.path.join(data_dir, "*.csv")) + glob.glob(os.path.join(data_dir, "*.xlsx"))
# language_results = {}

# for file_path in all_files:
#     filename = os.path.basename(file_path)
#     lang_name, config = determine_lang_info(filename)
    
#     # if not lang_name or lang_name != "Nagamese":
#     if not lang_name or lang_name != "Kokborok":
#         continue 
        
#     src_code = config["madlad"]
#     xlit_code = config["xlit"]
    
#     print(f"\nProcessing {lang_name} Dataset Conversion Logic...")   
#     try:
#         df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)     
#         src_col, en_col = detect_columns(df, filename)
#         df = df[[src_col, en_col]].dropna().astype(str)
        
#         raw_src_sentences = df[src_col].tolist()
#         ref_sentences = df[en_col].tolist()
        
#         native_assamese_sentences = run_transliteration(raw_src_sentences, xlit_code)
        
#         print(f"Translating {len(native_assamese_sentences)} script-mapped sentences using MADLAD...")
#         hyp_sentences = []
        
#         for batch_hyp in tqdm(translate_batch_generator(native_assamese_sentences, tgt_code="en", batch_size=BATCH_SIZE), 
#                               total=(len(native_assamese_sentences) + (BATCH_SIZE - 1)) // BATCH_SIZE, desc="Batches"):
#             hyp_sentences.extend(batch_hyp)
                
#         # Save predictions alongside matching columns
#         # prediction_df = pd.DataFrame({
#         #     'Nagamese_Roman_Original': raw_src_sentences,
#         #     'Nagamese_Assamese_Script': native_assamese_sentences,
#         #     'Ground_Truth_English': ref_sentences,
#         #     'Prediction': hyp_sentences
#         # })

#         prediction_df = pd.DataFrame({
#             'Tagin_Roman_Original': raw_src_sentences,
#             'Tagin_Bodo_Script': native_assamese_sentences,
#             'Ground_Truth_English': ref_sentences,
#             'Prediction': hyp_sentences
#         })
        
#         # prediction_output_path = f"{os.path.splitext(filename)[0]}_nagamese_predictions.csv"
#         prediction_output_path = f"{os.path.splitext(filename)[0]}_kokborok_predictions.csv"
#         prediction_df.to_csv(prediction_output_path, index=False)
#         print(f"Predictions saved to: {prediction_output_path}")  
        
#         # Computing metrics block
#         bleu_ref = [[ref] for ref in ref_sentences]
#         scores = {"BLEU": None, "chrF": None, "ROUGE-1": None, "ROUGE-L": None, "METEOR": None, "BERTScore-F1": None}
        
#         try: scores["BLEU"] = round(bleu_metric.compute(predictions=hyp_sentences, references=bleu_ref)['bleu'] * 100, 2)
#         except Exception as me: print(f" [Error] BLEU fail: {me}")
#         try: scores["chrF"] = round(chrf_metric.compute(predictions=hyp_sentences, references=bleu_ref)['score'], 2)
#         except Exception as me: print(f" [Error] chrF fail: {me}")
#         try:
#             rouge = rouge_metric.compute(predictions=hyp_sentences, references=ref_sentences)
#             scores["ROUGE-1"], scores["ROUGE-L"] = round(rouge['rouge1'] * 100, 2), round(rouge['rougeL'] * 100, 2)
#         except Exception as me: print(f" [Error] ROUGE fail: {me}")
#         try: scores["METEOR"] = round(meteor_metric.compute(predictions=hyp_sentences, references=ref_sentences)['meteor'] * 100, 2)
#         except Exception as me: print(f" [Error] METEOR fail: {me}")
#         try:
#             bertscore = bertscore_metric.compute(predictions=hyp_sentences, references=ref_sentences, lang="en", batch_size=BATCH_SIZE, device=device)
#             scores["BERTScore-F1"] = round(sum(bertscore['f1']) / len(bertscore['f1']) * 100, 2)
#         except Exception as me: print(f" [Error] BERTScore fail: {me}")
            
#         language_results[lang_name] = scores
#         save_incremental_results(language_results, output_path)
        
#     except Exception as e:
#         print(f" Critical error occurred: {e}")
#     finally:
#         gc.collect()
#         if device == "cuda": torch.cuda.empty_cache()

# if language_results:
#     print("\n" + "="*60 + "\n FINAL EVALUATION SUMMARY \n" + "="*60)
#     print(pd.DataFrame.from_dict(language_results, orient='index').to_markdown())